# Week 4 — Data Structures Deep Dive
### Python for Blockchain Analytics | Phase 1

---

This lesson goes much deeper than the intro lesson.
Each structure gets its full range of methods, edge cases, patterns,
and blockchain-native examples.

**Sections:**
1. Lists — all methods, sorting tricks, copying, joining, 2D lists
2. Tuples — advanced patterns, namedtuples, structured records
3. Dictionaries — all methods, merging, defaultdict, Counter, ordered operations
4. Sets — frozensets, performance, advanced set math on holder data
5. Nesting — real API shapes, deeply nested access, flattening
6. Choosing + Performance — when each structure wins and why

---

---
## Section 1 — Lists: Full Deep Dive

### 1.1 All the methods you need to know

In [23]:
# Every list method demonstrated on blockchain data

tx_hashes = ["0xaaa", "0xbbb", "0xccc", "0xddd"]
prices    = [3100.0, 3400.0, 3247.85, 3180.5, 3310.0]

# ── Adding ──
tx_hashes.append("0xeee")          # add to end
tx_hashes.insert(0, "0xfirst")     # insert at index
tx_hashes.extend(["0xfff", "0xggg"]) # add multiple (from another list)
print("After adds:", tx_hashes)

# ── Removing ──
tx_hashes.remove("0xfirst")        # remove by VALUE (first match)
popped = tx_hashes.pop()           # remove + return LAST item
popped2 = tx_hashes.pop(1)         # remove + return item at INDEX
print("After removes:", tx_hashes)
print("Popped:", popped, popped2)

# ── Searching ──
prices = [3100.0, 3400.0, 3247.85, 3180.5, 3310.0, 3400.0]
print("Count of 3400.0:", prices.count(3400.0))    # how many times
print("Index of 3247.85:", prices.index(3247.85))  # first position
print("3310.0 in list:", 3310.0 in prices)         # membership

# ── Ordering ──
prices_copy = prices.copy()
prices_copy.sort()                  # sort IN PLACE (modifies the list)
print("Sorted in place:", prices_copy)

prices_copy.sort(reverse=True)      # descending
print("Sorted desc:", prices_copy)

sorted_prices = sorted(prices)      # returns NEW list, original unchanged
print("Original unchanged:", prices)
print("New sorted:", sorted_prices)

# ── Other ──
prices.reverse()                    # reverse IN PLACE
print("Reversed:", prices)

prices_copy2 = prices.copy()        # shallow copy
print("Are they same object?", prices is prices_copy2)   # False ✓

print("Length:", len(prices))
prices.clear()                      # remove all items
print("After clear:", prices)

After adds: ['0xfirst', '0xaaa', '0xbbb', '0xccc', '0xddd', '0xeee', '0xfff', '0xggg']
After removes: ['0xaaa', '0xccc', '0xddd', '0xeee', '0xfff']
Popped: 0xggg 0xbbb
Count of 3400.0: 2
Index of 3247.85: 2
3310.0 in list: True
Sorted in place: [3100.0, 3180.5, 3247.85, 3310.0, 3400.0, 3400.0]
Sorted desc: [3400.0, 3400.0, 3310.0, 3247.85, 3180.5, 3100.0]
Original unchanged: [3100.0, 3400.0, 3247.85, 3180.5, 3310.0, 3400.0]
New sorted: [3100.0, 3180.5, 3247.85, 3310.0, 3400.0, 3400.0]
Reversed: [3400.0, 3310.0, 3180.5, 3247.85, 3400.0, 3100.0]
Are they same object? False
Length: 6
After clear: []


### 1.2 Sorting with custom keys

The most powerful sorting technique — sort by any field of complex objects.

In [27]:
# Sorting a list of dicts — the key= argument
# SQL parallel: ORDER BY volume_usd DESC, symbol ASC

tokens = [
    {"symbol": "UNI",  "price": 12.84,  "volume_24h": 180_000_000,  "change": -1.2},
    {"symbol": "ETH",  "price": 3247.85,"volume_24h": 15_800_000_000,"change":  2.4},
    {"symbol": "AAVE", "price": 98.50,  "volume_24h": 95_000_000,    "change":  3.1},
    {"symbol": "ARB",  "price": 1.24,   "volume_24h": 420_000_000,   "change": -0.5},
    {"symbol": "CRV",  "price": 0.48,   "volume_24h": 130_000_000,   "change":  1.8},
]

# Sort by price (ascending)
by_price = sorted(tokens, key=lambda t: t["price"])
print("By price asc:", [t["symbol"] for t in by_price])

# Sort by volume (descending)
by_volume = sorted(tokens, key=lambda t: t["volume_24h"], reverse=True)
print("By volume desc:", [t["symbol"] for t in by_volume])

# Sort by 24h change (best performers first)
by_change = sorted(tokens, key=lambda t: t["change"], reverse=True)
print("Best 24h perf:", [f"{t['symbol']}({t['change']:+.1f}%)" for t in by_change])

# Multi-key sort: first by change direction (positive first), then by volume
# SQL: ORDER BY (CASE WHEN change > 0 THEN 0 ELSE 1 END), volume_24h DESC
by_multi = sorted(tokens, key=lambda t: (-1 if t["change"] > 0 else 1, -t["volume_24h"]))
print("Positive first, then by volume:", [t["symbol"] for t in by_multi])

by_price

By price asc: ['CRV', 'ARB', 'UNI', 'AAVE', 'ETH']
By volume desc: ['ETH', 'ARB', 'UNI', 'CRV', 'AAVE']
Best 24h perf: ['AAVE(+3.1%)', 'ETH(+2.4%)', 'CRV(+1.8%)', 'ARB(-0.5%)', 'UNI(-1.2%)']
Positive first, then by volume: ['ETH', 'CRV', 'AAVE', 'ARB', 'UNI']


[{'symbol': 'CRV', 'price': 0.48, 'volume_24h': 130000000, 'change': 1.8},
 {'symbol': 'ARB', 'price': 1.24, 'volume_24h': 420000000, 'change': -0.5},
 {'symbol': 'UNI', 'price': 12.84, 'volume_24h': 180000000, 'change': -1.2},
 {'symbol': 'AAVE', 'price': 98.5, 'volume_24h': 95000000, 'change': 3.1},
 {'symbol': 'ETH', 'price': 3247.85, 'volume_24h': 15800000000, 'change': 2.4}]

### 1.3 Copying — shallow vs deep

This is a common source of bugs. Know the difference.

In [31]:
import copy
# The shallow copy problem — nested structures are STILL shared
wallets = [
    {"address": "0xAlice", "balance": 5.0},
    {"address": "0xBob",   "balance": 2.3},
]
shallow_wallets = wallets.copy()
shallow_wallets[0]["balance"] = 999.0  # modifies the INNER dict
print("Original wallet affected?", wallets[0]["balance"])  # 999.0 — PROBLEM!

# Deep copy — fully independent, no shared references
wallets = [
    {"address": "0xAlice", "balance": 5.0},
    {"address": "0xBob",   "balance": 2.3},
]
deep_wallets = copy.deepcopy(wallets)
deep_wallets[0]["balance"] = 999.0
print("Original safe with deepcopy?", wallets[0]["balance"])  # 5.0 — SAFE ✓


Original wallet affected? 999.0
Original safe with deepcopy? 5.0


In [32]:
import copy

# The problem: assignment doesn't copy — it creates another reference
original = [3100.0, 3247.85, 3400.0]
alias = original          # NOT a copy — same object!
alias.append(9999.0)
print("Original affected?", original)   # YES — both point to same list

# Shallow copy — works for flat lists (no nested structures)
original = [3100.0, 3247.85, 3400.0]
shallow = original.copy()        # or list(original) or original[:]
shallow.append(9999.0)
print("Original safe?", original)       # YES — different objects
print("Shallow copy:", shallow)

# The shallow copy problem — nested structures are STILL shared
wallets = [
    {"address": "0xAlice", "balance": 5.0},
    {"address": "0xBob",   "balance": 2.3},
]
shallow_wallets = wallets.copy()
shallow_wallets[0]["balance"] = 999.0  # modifies the INNER dict
print("Original wallet affected?", wallets[0]["balance"])  # 999.0 — PROBLEM!

# Deep copy — fully independent, no shared references
wallets = [
    {"address": "0xAlice", "balance": 5.0},
    {"address": "0xBob",   "balance": 2.3},
]
deep_wallets = copy.deepcopy(wallets)
deep_wallets[0]["balance"] = 999.0
print("Original safe with deepcopy?", wallets[0]["balance"])  # 5.0 — SAFE ✓

print("""
Rule of thumb:
  Flat list (ints, floats, strings)  → .copy() is fine
  List of dicts / nested lists       → use copy.deepcopy()
""")

Original affected? [3100.0, 3247.85, 3400.0, 9999.0]
Original safe? [3100.0, 3247.85, 3400.0]
Shallow copy: [3100.0, 3247.85, 3400.0, 9999.0]
Original wallet affected? 999.0
Original safe with deepcopy? 5.0

Rule of thumb:
  Flat list (ints, floats, strings)  → .copy() is fine
  List of dicts / nested lists       → use copy.deepcopy()



### 1.4 Joining, splitting, zipping

Converting between lists and other types — essential for API data.

In [40]:
# join — list of strings → one string
# Inverse of split()
token_symbols = ["ETH", "BTC", "UNI", "AAVE"]
csv_line      = ",".join(token_symbols)
pipe_line     = " | ".join(token_symbols)
print("CSV:  ", csv_line)
print("Pipe: ", pipe_line)

# split — string → list (parsing CSV or API params)
raw_csv    = "ETH,BTC,UNI,AAVE,ARB"
tokens     = raw_csv.split(",")
print("Split:", tokens)

# Parsing a comma-separated watchlist from a config
config_watchlist = "0xAlice, 0xBob, 0xCarol, 0xDave"
wallets = [w.strip() for w in config_watchlist.split(",")]
print("Parsed wallets:", wallets)

# zip — combine two lists element-by-element
# SQL parallel: JOIN two single-column result sets on row position
symbols  = ["ETH", "BTC", "UNI", "AAVE"]
prices   = [3247.85, 67412.0, 12.84, 98.50]
volumes  = [15_800_000_000, 32_000_000_000, 180_000_000, 95_000_000]

# zip two lists into pairs
price_map = dict(zip(symbols, prices))
print("Price map:", price_map)

# zip three lists
for sym, price, vol in zip(symbols, prices, volumes):
    print(f"  {sym:5} ${price:>10,.2f}   Vol: ${vol/1e9:.1f}B")

# unzip — inverse of zip
pairs = [("ETH", 3247.85), ("BTC", 67412.0), ("UNI", 12.84)]
syms, prs = zip(*pairs)   # * unpacks the list of tuples
print("Unzipped symbols:", syms)
print("Unzipped prices:", prs)

CSV:   ETH,BTC,UNI,AAVE
Pipe:  ETH | BTC | UNI | AAVE
Split: ['ETH', 'BTC', 'UNI', 'AAVE', 'ARB']
Parsed wallets: ['0xAlice', '0xBob', '0xCarol', '0xDave']
Price map: {'ETH': 3247.85, 'BTC': 67412.0, 'UNI': 12.84, 'AAVE': 98.5}
  ETH   $  3,247.85   Vol: $15.8B
  BTC   $ 67,412.00   Vol: $32.0B
  UNI   $     12.84   Vol: $0.2B
  AAVE  $     98.50   Vol: $0.1B
Unzipped symbols: ('ETH', 'BTC', 'UNI')
Unzipped prices: (3247.85, 67412.0, 12.84)


### 1.5 2D lists — block data matrices

In [48]:
# 2D list — list of lists
# Use case: daily price data for multiple tokens
#           rows = days, columns = [ETH, BTC, UNI, AAVE]

price_matrix = [
    # ETH       BTC         UNI     AAVE
    [3100.0,  65_000.0,   10.50,  85.00],   # Day 1
    [3180.5,  66_200.0,   11.20,  88.50],   # Day 2
    [3247.85, 67_412.0,   12.84,  98.50],   # Day 3
    [3310.0,  67_800.0,   13.10,  101.0],   # Day 4
    [3290.0,  67_100.0,   12.75,  99.00],   # Day 5
]

TOKENS = ["ETH", "BTC", "UNI", "AAVE"]
DAYS   = ["Mon", "Tue", "Wed", "Thu", "Fri"]

# Access: matrix[row][col]
print("ETH on Day 3:", price_matrix[2][0])     # 3247.85
print("AAVE on Day 5:", price_matrix[4][3])    # 99.00

# Print table
print(f"\n  {'Day':<5}", end="")
for t in TOKENS:
    print(f"  {t:>10}", end="")
print()
print("  " + "-" * 50)
for day, row in zip(DAYS, price_matrix):
    print(f"  {day:<5}", end="")
    for price in row:
        print(f"  ${price:>10,.2f}", end="")
    print()

# Column operation — get all ETH prices (column 0)
eth_prices = [row[0] for row in price_matrix]
print(f"\nETH prices: {eth_prices}")
print(f"ETH avg: ${sum(eth_prices)/len(eth_prices):,.2f}")

# Row operation — get all prices on Day 3 (row 2)
day3_prices = price_matrix[2]
print(f"\nDay 3 prices: {day3_prices}")

ETH on Day 3: 3247.85
AAVE on Day 5: 99.0

  Day           ETH         BTC         UNI        AAVE
  --------------------------------------------------
  Mon    $  3,100.00  $ 65,000.00  $     10.50  $     85.00
  Tue    $  3,180.50  $ 66,200.00  $     11.20  $     88.50
  Wed    $  3,247.85  $ 67,412.00  $     12.84  $     98.50
  Thu    $  3,310.00  $ 67,800.00  $     13.10  $    101.00
  Fri    $  3,290.00  $ 67,100.00  $     12.75  $     99.00

ETH prices: [3100.0, 3180.5, 3247.85, 3310.0, 3290.0]
ETH avg: $3,225.67

Day 3 prices: [3247.85, 67412.0, 12.84, 98.5]


---
## Section 2 — Tuples: Full Deep Dive

### 2.1 Why tuples exist — and when to choose them

In [50]:
# Tuples signal INTENT — this data is fixed and should not change
# A confirmed transaction: immutable by design (just like on-chain data)

# Tuple vs list — same data, different semantics
tx_as_list  = ["0xabc", 19_847_293, 1_714_000_000, "success"]  # mutable — feels wrong
tx_as_tuple = ("0xabc", 19_847_293, 1_714_000_000, "success")  # immutable — correct

# Lists are for homogeneous sequences (all same type of thing)
prices = [3100.0, 3180.5, 3247.85]            # all prices

# Tuples are for heterogeneous records (different fields of one thing)
price_point = (3247.85, 1_714_000_000, "ETH") # (price, timestamp, token)

# Proof of immutability
try:
    tx_as_tuple[0] = "0xnew"
except TypeError as e:
    print(f"Can't modify tuple: {e}")

# One-element tuple — needs trailing comma!
single = (42,)         # this is a tuple
not_tuple = (42)       # this is just an int in parentheses
print(type(single))    # <class 'tuple'>
print(type(not_tuple)) # <class 'int'>

Can't modify tuple: 'tuple' object does not support item assignment
<class 'tuple'>
<class 'int'>


In [54]:
# Tuple packing and unpacking — the full picture

# Packing — create a tuple from values
swap = "0xabc", 19_847_293, "USDC", 1000.0, "ETH", 0.3082  # no parens needed!

# Basic unpacking
tx_hash, block, token_in, amount_in, token_out, amount_out = swap
print(f"Swap: {amount_in} {token_in} → {amount_out} {token_out} at block {block:,}")

# Starred unpacking — capture the "rest"
first, *middle, last = (1, 2, 3, 4, 5)
print(f"First: {first}, Middle: {middle}, Last: {last}")

# Swap two variables (Python's elegant trick)
a, b = 10, 20
print(f"Before: a={a}, b={b}")
a, b = b, a           # swap without a temp variable
print(f"After:  a={a}, b={b}")

# Practical: swap token order for reverse lookup
token_in, token_out = "USDC", "ETH"
token_in, token_out = token_out, token_in
print(f"Reversed pair: {token_in} → {token_out}")

# Unpacking in for loops — process structured records cleanly
swap_events = [
    ("0xaaa", "USDC", 1000.0,  "ETH",  0.3082),
    ("0xbbb", "ETH",  2.5,     "USDC", 8119.63),
    ("0xccc", "UNI",  100,     "ETH",  0.3952),
]

print("\nSwap history:")
for tx_hash, t_in, amt_in, t_out, amt_out in swap_events:
    print(f"  {tx_hash} | {amt_in:>10,.4f} {t_in:5} → {amt_out:>10,.4f} {t_out}")

Swap: 1000.0 USDC → 0.3082 ETH at block 19,847,293
First: 1, Middle: [2, 3, 4], Last: 5
Before: a=10, b=20
After:  a=20, b=10
Reversed pair: ETH → USDC

Swap history:
  0xaaa | 1,000.0000 USDC  →     0.3082 ETH
  0xbbb |     2.5000 ETH   → 8,119.6300 USDC
  0xccc |   100.0000 UNI   →     0.3952 ETH


In [57]:
from collections import namedtuple

# namedtuple — a tuple where fields have names
# Best of both worlds: immutable like tuple, readable like dict

# Define the structure (like a schema)
SwapEvent = namedtuple("SwapEvent", [
    "tx_hash", "block_number", "token_in", "amount_in",
    "token_out", "amount_out", "gas_used"
])

TokenPrice = namedtuple("TokenPrice", ["symbol", "price_usd", "timestamp"])

# Create instances
swap = SwapEvent(
    tx_hash="0xabc123",
    block_number=19_847_293,
    token_in="USDC",
    amount_in=1000.0,
    token_out="ETH",
    amount_out=0.3082,
    gas_used=148_320
)

# Access by name — much clearer than swap[2]
print(f"Token in:  {swap.token_in}")
print(f"Amount in: {swap.amount_in}")
print(f"Gas used:  {swap.gas_used:,}")

# Still works as a tuple
tx_hash, block, *_ = swap
print(f"Hash: {tx_hash}, Block: {block:,}")

# Batch of named records
prices = [
    TokenPrice("ETH",  3247.85, 1_714_000_000),
    TokenPrice("BTC",  67412.0, 1_714_000_000),
    TokenPrice("UNI",  12.84,   1_714_000_000),
]

# Sort namedtuples just like dicts
sorted_prices = sorted(prices, key=lambda p: p.price_usd, reverse=True)
for p in sorted_prices:
    print(f"  {p.symbol:5} ${p.price_usd:>10,.2f}")

# Convert to dict when needed
swap_dict = swap._asdict()
print("\nAs dict:", dict(swap_dict))

Token in:  USDC
Amount in: 1000.0
Gas used:  148,320
Hash: 0xabc123, Block: 19,847,293
  BTC   $ 67,412.00
  ETH   $  3,247.85
  UNI   $     12.84

As dict: {'tx_hash': '0xabc123', 'block_number': 19847293, 'token_in': 'USDC', 'amount_in': 1000.0, 'token_out': 'ETH', 'amount_out': 0.3082, 'gas_used': 148320}


---
## Section 3 — Dictionaries: Full Deep Dive

### 3.1 All dict methods

In [59]:
# Every dict method on blockchain data

token = {
    "symbol":    "ETH",
    "price_usd": 3247.85,
    "chain":     "ethereum",
    "decimals":  18,
    "verified":  True,
}

# ── Reading ──
print(token["symbol"])                          # direct access — KeyError if missing
print(token.get("symbol"))                      # safe access — None if missing
print(token.get("volume", 0))                   # safe access with DEFAULT
print("symbol" in token)                        # key existence check
print("volume" not in token)                    # negative check

# ── All keys, values, items ──
print("Keys:  ", list(token.keys()))
print("Values:", list(token.values()))
print("Items: ", list(token.items()))           # list of (key, value) tuples

# ── Adding / updating ──
token["volume_24h"] = 15_800_000_000           # add new key
token["price_usd"]  = 3310.0                   # update existing
token.update({"rank": 1, "category": "L1"})    # update multiple at once
token.setdefault("launch_year", 2015)           # set ONLY if key doesn't exist
token.setdefault("price_usd", 0)               # does NOT overwrite existing
print("price_usd unchanged?", token["price_usd"])  # still 3310.0

# ── Removing ──
removed_val = token.pop("category")            # remove + return value
print("Removed:", removed_val)
token.pop("nonexistent", None)                  # safe pop with default — no error
last_item = token.popitem()                     # remove + return last (key, value)
print("Last item removed:", last_item)

# ── Copying ──
token_copy = token.copy()                       # shallow copy
print("Same object?", token is token_copy)      # False ✓

ETH
ETH
0
True
True
Keys:   ['symbol', 'price_usd', 'chain', 'decimals', 'verified']
Values: ['ETH', 3247.85, 'ethereum', 18, True]
Items:  [('symbol', 'ETH'), ('price_usd', 3247.85), ('chain', 'ethereum'), ('decimals', 18), ('verified', True)]
price_usd unchanged? 3310.0
Removed: L1
Last item removed: ('launch_year', 2015)
Same object? False


In [63]:
from collections import defaultdict, Counter

# defaultdict — dict that auto-creates missing keys
# Eliminates the pattern: if key not in d: d[key] = []

# Without defaultdict (old way — verbose)
type_counts = {}
tx_types = ["swap", "transfer", "swap", "liquidity", "swap", "transfer", "NFT"]
for t in tx_types:
    if t not in type_counts:
        type_counts[t] = 0
    type_counts[t] += 1

# With defaultdict(int) — auto-initialises to 0
type_counts = defaultdict(int)
for t in tx_types:
    type_counts[t] += 1    # no KeyError on first access!
print("Type counts:", dict(type_counts))

# defaultdict(list) — auto-initialises to []
# SQL parallel: GROUP BY wallet, collecting tx hashes per wallet
wallet_txs = defaultdict(list)
transactions = [
    {"wallet": "0xAlice", "hash": "0xaaa"},
    {"wallet": "0xBob",   "hash": "0xbbb"},
    {"wallet": "0xAlice", "hash": "0xccc"},
    {"wallet": "0xAlice", "hash": "0xddd"},
    {"wallet": "0xBob",   "hash": "0xeee"},
]
for tx in transactions:
    wallet_txs[tx["wallet"]].append(tx["hash"])

for wallet, hashes in wallet_txs.items():
    print(f"  {wallet}: {len(hashes)} txns — {hashes}")

# Counter — dict subclass for counting (perfect for blockchain analytics)
# SQL parallel: SELECT type, COUNT(*) FROM txns GROUP BY type ORDER BY COUNT(*) DESC
token_interactions = ["ETH", "USDC", "ETH", "UNI", "ETH", "USDC", "AAVE", "ETH", "UNI"]

counter = Counter(token_interactions)
print("\nToken interaction counts:", counter)
print("Most common (top 3):", counter.most_common(3))
print("ETH count:", counter["ETH"])

# Counter arithmetic
yesterday = Counter({"ETH": 5, "USDC": 3, "UNI": 2})
today     = Counter({"ETH": 3, "USDC": 5, "ARB": 4})
combined  = yesterday + today
growth    = today - yesterday   # only positive differences
print("Combined:", dict(combined))
print("Growth (today vs yesterday):", dict(growth))

Type counts: {'swap': 3, 'transfer': 2, 'liquidity': 1, 'NFT': 1}
  0xAlice: 3 txns — ['0xaaa', '0xccc', '0xddd']
  0xBob: 2 txns — ['0xbbb', '0xeee']

Token interaction counts: Counter({'ETH': 4, 'USDC': 2, 'UNI': 2, 'AAVE': 1})
Most common (top 3): [('ETH', 4), ('USDC', 2), ('UNI', 2)]
ETH count: 4
Combined: {'ETH': 8, 'USDC': 8, 'UNI': 2, 'ARB': 4}
Growth (today vs yesterday): {'USDC': 2, 'ARB': 4}


In [65]:
# Merging dicts — multiple approaches

base_token = {"symbol": "UNI", "chain": "ethereum", "decimals": 18}
price_data = {"price_usd": 12.84, "volume_24h": 180_000_000}
meta_data  = {"verified": True, "rank": 15, "chain": "optimism"}  # note: overlapping "chain"

# Method 1: update() — modifies IN PLACE
merged = base_token.copy()
merged.update(price_data)
merged.update(meta_data)         # "chain" overwritten by meta_data
print("update():", merged)

# Method 2: ** unpacking — creates new dict (Python 3.5+)
merged2 = {**base_token, **price_data, **meta_data}
print("** unpack:", merged2)

# Method 3: | operator — Python 3.9+ (cleanest)
merged3 = base_token | price_data | meta_data
print("| operator:", merged3)

# Last write wins for duplicate keys — be intentional about order
# Put the dict with HIGHER PRIORITY last
defaults  = {"decimals": 18, "verified": False, "chain": "ethereum"}
overrides = {"verified": True, "chain": "base"}    # these should win
final = {**defaults, **overrides}
print("\nMerge with override:", final)

update(): {'symbol': 'UNI', 'chain': 'optimism', 'decimals': 18, 'price_usd': 12.84, 'volume_24h': 180000000, 'verified': True, 'rank': 15}
** unpack: {'symbol': 'UNI', 'chain': 'optimism', 'decimals': 18, 'price_usd': 12.84, 'volume_24h': 180000000, 'verified': True, 'rank': 15}
| operator: {'symbol': 'UNI', 'chain': 'optimism', 'decimals': 18, 'price_usd': 12.84, 'volume_24h': 180000000, 'verified': True, 'rank': 15}

Merge with override: {'decimals': 18, 'verified': True, 'chain': 'base'}


In [66]:
# Dict as a lookup table — replacing long if/elif chains
# SQL parallel: JOIN to a reference/dimension table

# Instead of this:
def get_chain_name_verbose(chain_id):
    if chain_id == 1:      return "Ethereum"
    elif chain_id == 137:  return "Polygon"
    elif chain_id == 42161:return "Arbitrum"
    elif chain_id == 8453: return "Base"
    elif chain_id == 10:   return "Optimism"
    else:                  return "Unknown"

# Use this:
CHAIN_NAMES = {
    1:      "Ethereum",
    137:    "Polygon",
    42161:  "Arbitrum One",
    8453:   "Base",
    10:     "Optimism",
    56:     "BNB Chain",
    43114:  "Avalanche",
}

chain_ids = [1, 137, 42161, 99999, 8453]
for cid in chain_ids:
    name = CHAIN_NAMES.get(cid, "Unknown Chain")
    print(f"  Chain {cid:>6} → {name}")

  Chain      1 → Ethereum
  Chain    137 → Polygon
  Chain  42161 → Arbitrum One
  Chain  99999 → Unknown Chain
  Chain   8453 → Base


In [67]:
# Dict as a lookup table — replacing long if/elif chains
# SQL parallel: JOIN to a reference/dimension table

# Instead of this:
def get_chain_name_verbose(chain_id):
    if chain_id == 1:      return "Ethereum"
    elif chain_id == 137:  return "Polygon"
    elif chain_id == 42161:return "Arbitrum"
    elif chain_id == 8453: return "Base"
    elif chain_id == 10:   return "Optimism"
    else:                  return "Unknown"

# Use this:
CHAIN_NAMES = {
    1:      "Ethereum",
    137:    "Polygon",
    42161:  "Arbitrum One",
    8453:   "Base",
    10:     "Optimism",
    56:     "BNB Chain",
    43114:  "Avalanche",
}

chain_ids = [1, 137, 42161, 99999, 8453]
for cid in chain_ids:
    name = CHAIN_NAMES.get(cid, "Unknown Chain")
    print(f"  Chain {cid:>6} → {name}")

# Dict dispatch — call different functions based on event type
def handle_swap(event):       return f"Swap: {event.get('amount_in')} → {event.get('amount_out')}"
def handle_transfer(event):   return f"Transfer: {event.get('value')} ETH"
def handle_liquidity(event):  return f"LP: {event.get('token')} pool"

EVENT_HANDLERS = {
    "swap":      handle_swap,
    "transfer":  handle_transfer,
    "liquidity": handle_liquidity,
}

events = [
    {"type": "swap",      "amount_in": 1000, "amount_out": 0.308},
    {"type": "transfer",  "value": 2.5},
    {"type": "liquidity", "token": "ETH/USDC"},
    {"type": "unknown",   "data": "???"},
]

for event in events:
    handler = EVENT_HANDLERS.get(event["type"])
    if handler:
        print(handler(event))
    else:
        print(f"Unknown event type: {event['type']}")

  Chain      1 → Ethereum
  Chain    137 → Polygon
  Chain  42161 → Arbitrum One
  Chain  99999 → Unknown Chain
  Chain   8453 → Base
Swap: 1000 → 0.308
Transfer: 2.5 ETH
LP: ETH/USDC pool
Unknown event type: unknown


---
## Section 4 — Sets: Full Deep Dive

### 4.1 All set methods and operations

In [68]:
# All set methods on blockchain data

holders = {"0xAlice", "0xBob", "0xCarol", "0xDave"}

# ── Adding ──
holders.add("0xEve")              # add one element
holders.update(["0xFrank", "0xGrace"])  # add multiple
print("After adds:", holders)

# ── Removing ──
holders.remove("0xGrace")        # remove — KeyError if not found
holders.discard("0xNobody")      # remove — NO error if not found (safer)
popped = holders.pop()           # remove and return ARBITRARY element
print("Popped:", popped)
print("After removes:", holders)

# ── Checking ──
print("0xAlice in set:", "0xAlice" in holders)
print("Set length:", len(holders))

# ── Copying & clearing ──
backup = holders.copy()
print("Are they same?", holders is backup)  # False
# holders.clear()  # empties the set

After adds: {'0xDave', '0xGrace', '0xCarol', '0xEve', '0xAlice', '0xFrank', '0xBob'}
Popped: 0xDave
After removes: {'0xCarol', '0xEve', '0xAlice', '0xFrank', '0xBob'}
0xAlice in set: True
Set length: 5
Are they same? False


In [69]:
# All set operations — method AND operator syntax
a = {"0xAlice", "0xBob",   "0xCarol", "0xDave"}
b = {"0xBob",   "0xDave",  "0xEve",   "0xFrank"}

# Intersection — in BOTH sets
print("Intersection (&):", a & b)
print("Intersection (method):", a.intersection(b))

# Union — in EITHER set
print("Union (|):", a | b)

# Difference — in a but NOT in b
print("Difference (a-b):", a - b)
print("Difference (b-a):", b - a)

# Symmetric difference — in one but NOT both
print("Sym diff (^):", a ^ b)

# Subset / superset checks
small = {"0xAlice", "0xBob"}
print("small ⊆ a:", small.issubset(a))        # True
print("a ⊇ small:", a.issuperset(small))       # True
print("a and b disjoint:", a.isdisjoint(b))    # False (they share elements)

# In-place operations (modify set a directly)
a_copy = a.copy()
a_copy &= b    # keep only what's in both
print("a after &= b:", a_copy)

a_copy = a.copy()
a_copy |= b    # add all from b
print("a after |= b:", a_copy)

Intersection (&): {'0xDave', '0xBob'}
Intersection (method): {'0xDave', '0xBob'}
Union (|): {'0xDave', '0xCarol', '0xEve', '0xAlice', '0xFrank', '0xBob'}
Difference (a-b): {'0xAlice', '0xCarol'}
Difference (b-a): {'0xFrank', '0xEve'}
Sym diff (^): {'0xCarol', '0xEve', '0xAlice', '0xFrank'}
small ⊆ a: True
a ⊇ small: True
a and b disjoint: False
a after &= b: {'0xDave', '0xBob'}
a after |= b: {'0xDave', '0xCarol', '0xEve', '0xAlice', '0xFrank', '0xBob'}


In [70]:
# frozenset — immutable set (can be used as dict key or set element)

# Regular sets CANNOT be elements of other sets (unhashable)
try:
    set_of_sets = {{"0xAlice"}, {"0xBob"}}
except TypeError as e:
    print(f"Error: {e}")

# frozenset CAN be used as dict key or set element
protocol_a = frozenset({"0xAlice", "0xBob", "0xCarol"})
protocol_b = frozenset({"0xBob", "0xDave", "0xEve"})

# Use frozensets as dict keys — holder snapshot registry
snapshots = {
    protocol_a: {"protocol": "Uniswap", "block": 19_847_000},
    protocol_b: {"protocol": "Aave",    "block": 19_847_000},
}

for holder_set, meta in snapshots.items():
    overlap = holder_set & frozenset({"0xAlice", "0xBob", "0xXYZ"})
    print(f"{meta['protocol']}: {len(overlap)} matching holders")

# frozenset supports all read operations
print("0xAlice in protocol_a:", "0xAlice" in protocol_a)
print("Overlap:", protocol_a & protocol_b)

Error: unhashable type: 'set'
Uniswap: 2 matching holders
Aave: 1 matching holders
0xAlice in protocol_a: True
Overlap: frozenset({'0xBob'})


In [71]:
# Performance — why sets matter for large blockchain datasets
import time

# Scenario: checking 10,000 wallets against a 50,000-wallet blacklist

# Generate sample data
blacklist_list = [f"0xWallet{i:05d}" for i in range(50_000)]
blacklist_set  = set(blacklist_list)
wallets_to_check = [f"0xWallet{i:05d}" for i in range(0, 20_000, 2)]

# Check with LIST — O(n) per lookup → O(n*m) total
start = time.time()
hits_list = [w for w in wallets_to_check if w in blacklist_list]
list_time = time.time() - start

# Check with SET — O(1) per lookup → O(m) total
start = time.time()
hits_set = [w for w in wallets_to_check if w in blacklist_set]
set_time = time.time() - start

print(f"List lookup: {list_time:.3f}s  ({len(hits_list)} hits)")
print(f"Set lookup:  {set_time:.4f}s  ({len(hits_set)} hits)")
print(f"Set is {list_time/set_time:.0f}x faster")
print("""
Rule: if you're checking membership (x in collection) more than once,
convert to a set first. This matters enormously at blockchain scale
(millions of addresses, thousands of blocks).
""")

List lookup: 4.667s  (10000 hits)
Set lookup:  0.0038s  (10000 hits)
Set is 1215x faster

Rule: if you're checking membership (x in collection) more than once,
convert to a set first. This matters enormously at blockchain scale
(millions of addresses, thousands of blocks).



---
## Section 5 — Nesting: Real-World Blockchain API Shapes

### 5.1 Real API response structures

In [72]:
# This is what a real Etherscan transaction list response looks like
# (simplified but structurally accurate)

etherscan_response = {
    "status": "1",
    "message": "OK",
    "result": [
        {
            "blockNumber": "19847293",
            "hash": "0xabc123def456",
            "from": "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
            "to":   "0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984",
            "value": "1500000000000000000",
            "gas": "21000",
            "gasPrice": "20000000000",
            "gasUsed": "21000",
            "isError": "0",
            "txreceipt_status": "1",
            "input": "0x",
            "contractAddress": "",
            "confirmations": "142",
        },
        {
            "blockNumber": "19847150",
            "hash": "0xdef456abc123",
            "from": "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
            "to":   "0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D",
            "value": "500000000000000000",
            "gas": "200000",
            "gasPrice": "18000000000",
            "gasUsed": "148320",
            "isError": "0",
            "txreceipt_status": "1",
            "input": "0x38ed1739...",   # encoded swap call
            "contractAddress": "",
            "confirmations": "285",
        },
    ]
}

# ── Navigating nested structure ──
status   = etherscan_response["status"]
tx_list  = etherscan_response["result"]
tx_count = len(tx_list)
print(f"API status: {status} | Transactions: {tx_count}")

# ── Processing each transaction ──
print("\nTransaction breakdown:")
for tx in tx_list:
    block    = int(tx["blockNumber"])
    value    = int(tx["value"]) / 10**18
    gas_cost = int(tx["gasUsed"]) * int(tx["gasPrice"]) / 10**18
    is_swap  = tx["input"] != "0x"    # non-empty input = contract call
    tx_type  = "🔄 Swap" if is_swap else "➡️  Transfer"

    print(f"  Block {block:,} | {value:.4f} ETH | Gas: {gas_cost:.6f} ETH | {tx_type}")

API status: 1 | Transactions: 2

Transaction breakdown:
  Block 19,847,293 | 1.5000 ETH | Gas: 0.000420 ETH | ➡️  Transfer
  Block 19,847,150 | 0.5000 ETH | Gas: 0.002670 ETH | 🔄 Swap


In [73]:
# Deeply nested — DeFi protocol analytics response
# (similar shape to what TheGraph or custom indexers return)

protocol_data = {
    "protocol": "Uniswap V3",
    "chain": "ethereum",
    "stats": {
        "tvl_usd":      5_200_000_000,
        "volume_24h":   1_800_000_000,
        "fees_24h":     5_400_000,
        "pool_count":   8_432,
    },
    "top_pools": [
        {
            "address": "0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640",
            "token0": {"symbol": "USDC", "address": "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"},
            "token1": {"symbol": "ETH",  "address": "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2"},
            "fee_tier": 500,
            "tvl_usd":  800_000_000,
            "volume_24h": 420_000_000,
            "price": 3247.85,
        },
        {
            "address": "0x8ad599c3A0ff1De082011EFDDc58f1908eb6e6D8",
            "token0": {"symbol": "USDC", "address": "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"},
            "token1": {"symbol": "ETH",  "address": "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2"},
            "fee_tier": 3000,
            "tvl_usd":  450_000_000,
            "volume_24h": 180_000_000,
            "price": 3247.12,
        },
        {
            "address": "0x4e68Ccd3E89f51C3074ca5072bbAC773960dFa36",
            "token0": {"symbol": "ETH",  "address": "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2"},
            "token1": {"symbol": "USDT", "address": "0xdAC17F958D2ee523a2206206994597C13D831ec7"},
            "fee_tier": 3000,
            "tvl_usd":  320_000_000,
            "volume_24h": 95_000_000,
            "price": 3248.10,
        },
    ],
    "historical_tvl": {
        "2024-01-01": 4_100_000_000,
        "2024-02-01": 4_500_000_000,
        "2024-03-01": 4_900_000_000,
        "2024-04-01": 5_200_000_000,
    }
}

# ── Deep access patterns ──
# Level 1
print("Protocol:", protocol_data["protocol"])

# Level 2
print("TVL:", f"${protocol_data['stats']['tvl_usd']/1e9:.1f}B")

# Level 3 — into nested list then nested dict
top_pool_pair = (
    protocol_data["top_pools"][0]["token0"]["symbol"],
    protocol_data["top_pools"][0]["token1"]["symbol"],
)
print("Top pool pair:", "/".join(top_pool_pair))

# ── Iterating nested structures ──
print("\nPool breakdown:")
for pool in protocol_data["top_pools"]:
    pair     = f"{pool['token0']['symbol']}/{pool['token1']['symbol']}"
    fee_pct  = pool["fee_tier"] / 10_000
    apr_est  = (pool["volume_24h"] * fee_pct * 365) / pool["tvl_usd"] * 100
    print(f"  {pair:10} | Fee: {fee_pct:.2%} | TVL: ${pool['tvl_usd']/1e6:.0f}M "
          f"| Vol: ${pool['volume_24h']/1e6:.0f}M | Est APR: {apr_est:.1f}%")

# ── Historical data from nested dict ──
print("\nTVL growth:")
tvl_history = protocol_data["historical_tvl"]
dates = sorted(tvl_history.keys())
for i, date in enumerate(dates):
    tvl = tvl_history[date]
    if i > 0:
        prev = tvl_history[dates[i-1]]
        growth = (tvl - prev) / prev * 100
        print(f"  {date}: ${tvl/1e9:.2f}B ({growth:+.1f}%)")
    else:
        print(f"  {date}: ${tvl/1e9:.2f}B")

Protocol: Uniswap V3
TVL: $5.2B
Top pool pair: USDC/ETH

Pool breakdown:
  USDC/ETH   | Fee: 5.00% | TVL: $800M | Vol: $420M | Est APR: 958.1%
  USDC/ETH   | Fee: 30.00% | TVL: $450M | Vol: $180M | Est APR: 4380.0%
  ETH/USDT   | Fee: 30.00% | TVL: $320M | Vol: $95M | Est APR: 3250.8%

TVL growth:
  2024-01-01: $4.10B
  2024-02-01: $4.50B (+9.8%)
  2024-03-01: $4.90B (+8.9%)
  2024-04-01: $5.20B (+6.1%)


In [74]:
# Safe deep access — handling missing keys in nested structures

messy_data = {
    "token_a": {"price": 3247.85, "metadata": {"verified": True}},
    "token_b": {"price": 12.84},                  # no metadata key
    "token_c": {"metadata": {"verified": False}},  # no price key
}

# ❌ Unsafe — crashes if key is missing
# print(messy_data["token_b"]["metadata"]["verified"])  # KeyError!

# ✅ Option 1: Chained .get() with defaults
for token, data in messy_data.items():
    price    = data.get("price", 0)
    metadata = data.get("metadata", {})
    verified = metadata.get("verified", False)
    print(f"  {token}: price=${price:.2f}, verified={verified}")

# ✅ Option 2: try/except for complex nesting
for token, data in messy_data.items():
    try:
        verified = data["metadata"]["verified"]
    except KeyError:
        verified = False
    print(f"  {token}: verified={verified}")

  token_a: price=$3247.85, verified=True
  token_b: price=$12.84, verified=False
  token_c: price=$0.00, verified=False
  token_a: verified=True
  token_b: verified=False
  token_c: verified=False


In [75]:
# Flattening nested structures
# Often you need to collapse deeply nested data into a flat list of dicts

# Deeply nested: protocol → pools → daily snapshots
nested = {
    "Uniswap": {
        "ETH/USDC": [
            {"date": "2024-04-01", "volume": 420_000_000, "tvl": 800_000_000},
            {"date": "2024-04-02", "volume": 380_000_000, "tvl": 810_000_000},
        ],
        "ETH/USDT": [
            {"date": "2024-04-01", "volume": 95_000_000,  "tvl": 320_000_000},
            {"date": "2024-04-02", "volume": 110_000_000, "tvl": 325_000_000},
        ],
    },
    "Curve": {
        "3pool": [
            {"date": "2024-04-01", "volume": 210_000_000, "tvl": 500_000_000},
            {"date": "2024-04-02", "volume": 195_000_000, "tvl": 495_000_000},
        ],
    },
}

# Flatten to list of dicts (like a SQL result set)
flat_rows = []
for protocol, pools in nested.items():
    for pool_name, snapshots in pools.items():
        for snap in snapshots:
            flat_rows.append({
                "protocol": protocol,
                "pool":     pool_name,
                "date":     snap["date"],
                "volume":   snap["volume"],
                "tvl":      snap["tvl"],
            })

# Now it looks like a SQL result — easy to filter/sort/aggregate
print(f"Flattened: {len(flat_rows)} rows")
print(f"{'Protocol':<10} {'Pool':<12} {'Date':<12} {'Volume':>14} {'TVL':>14}")
print("-" * 65)
for row in flat_rows:
    print(f"{row['protocol']:<10} {row['pool']:<12} {row['date']:<12} "
          f"${row['volume']:>13,.0f} ${row['tvl']:>13,.0f}")

Flattened: 6 rows
Protocol   Pool         Date                 Volume            TVL
-----------------------------------------------------------------
Uniswap    ETH/USDC     2024-04-01   $  420,000,000 $  800,000,000
Uniswap    ETH/USDC     2024-04-02   $  380,000,000 $  810,000,000
Uniswap    ETH/USDT     2024-04-01   $   95,000,000 $  320,000,000
Uniswap    ETH/USDT     2024-04-02   $  110,000,000 $  325,000,000
Curve      3pool        2024-04-01   $  210,000,000 $  500,000,000
Curve      3pool        2024-04-02   $  195,000,000 $  495,000,000


---
## Section 6 — Choosing the Right Structure + Performance

### 6.1 Decision framework

In [76]:
# Decision framework as code + examples

print("""
CHOOSING A DATA STRUCTURE
─────────────────────────────────────────────────────────────────

QUESTION                          → ANSWER          → STRUCTURE
──────────────────────────────────────────────────────────────────
Do you need key→value lookup?     → Yes             → dict
Is order important?               → No  + unique    → set
Is it a fixed record of fields?   → Yes             → tuple / namedtuple
Do you need to check membership   
  on a large collection often?    → Yes             → set
Is the data mutable?              → No              → tuple / frozenset
                                  → Yes             → list / dict / set
Is it a sequence of same-type?    → Yes             → list
Is it a single structured record? → Yes             → dict or namedtuple
Is it a result set (rows)?        → Yes             → list of dicts
Do you need fast deduplication?   → Yes             → set (then list if needed)
──────────────────────────────────────────────────────────────────
""")

# Practical examples of each decision
# 1. "I need to look up a token's price by symbol"
token_prices = {"ETH": 3247.85, "BTC": 67412.0}   # → dict

# 2. "I have 1M wallet addresses and need to check membership 10k times"
blacklist = set(open("/dev/null").read().split())   # → set (fast O(1) lookup)
blacklist = {"0xScam1", "0xScam2"}                 # → set

# 3. "I'm storing a confirmed swap event that should never change"
from collections import namedtuple
Swap = namedtuple("Swap", ["hash", "block", "token_in", "amount_in", "token_out", "amount_out"])
swap = Swap("0xabc", 19_847_293, "USDC", 1000.0, "ETH", 0.3082)  # → namedtuple

# 4. "I'm building a list of all Uniswap swaps in the last hour"
recent_swaps = []   # → list (ordered, mutable, may have duplicates)

# 5. "I want unique protocols a wallet has interacted with"
protocols_used = set()   # → set (unique, unordered)

# 6. "I'm storing a query result from my analytics pipeline"
result_rows = [    # → list of dicts
    {"wallet": "0xAlice", "volume": 125_000, "tx_count": 42},
    {"wallet": "0xBob",   "volume": 87_500,  "tx_count": 28},
]

print("Decision examples demonstrated ✓")


CHOOSING A DATA STRUCTURE
─────────────────────────────────────────────────────────────────

QUESTION                          → ANSWER          → STRUCTURE
──────────────────────────────────────────────────────────────────
Do you need key→value lookup?     → Yes             → dict
Is order important?               → No  + unique    → set
Is it a fixed record of fields?   → Yes             → tuple / namedtuple
Do you need to check membership   
  on a large collection often?    → Yes             → set
Is the data mutable?              → No              → tuple / frozenset
                                  → Yes             → list / dict / set
Is it a sequence of same-type?    → Yes             → list
Is it a single structured record? → Yes             → dict or namedtuple
Is it a result set (rows)?        → Yes             → list of dicts
Do you need fast deduplication?   → Yes             → set (then list if needed)
──────────────────────────────────────────────────────────────────



## Summary of the full deep-dive

| Structure | Key extra methods | When it shines |
|-----------|-----------------|----------------|
| `list` | `.extend()`, `.sort(key=)`, `sorted(key=)`, `.copy()`, `copy.deepcopy()`, `zip()`, `.join()` | Ordered sequences, query results, price history |
| `tuple` | Unpacking, `*rest`, `namedtuple` | Fixed records, function returns, dict keys |
| `dict` | `defaultdict`, `Counter`, `.setdefault()`, `.update()`, `\|` merge, dispatch table | Key-value lookup, API responses, GROUP BY aggregation |
| `set` | `.add()`, `.discard()`, `frozenset`, all operators, performance | Deduplication, membership, holder overlap |
| Nesting | chained `.get()`, `try/except`, flattening | Real API shapes, protocol analytics |

---

**Next step:** `lesson_advanced.ipynb` digs into 5 applied projects that combine all these structures together on real-world blockchain problems.

Then hit `exercises_advanced.py` for the hard problems.